In [11]:
# Load data from the spreadsheet
import sys
import pandas as pd
from pathlib import Path
import openpyxl

from loguru import logger
logger.remove()
logger.add(sys.stderr, level="ERROR")

filename = Path(r"C:\Users\crdig\Downloads\ENOSCEN_APMay 08 2025 15_43_30.csv")
assert filename.exists(), f"File {filename} does not exist."

# Import the data
if filename.suffix == ".csv":
    df_raw = pd.read_csv(filename, header=0, dtype=object)
elif filename.suffix == ".xlsx":
    df_raw = pd.read_excel(filename, sheet_name="Sheet1", skiprows=0, usecols="A:EL", header=None, dtype=object)
# df_raw

In [12]:
# Categorize the data

# Iterate through each row. Create a new column "Assembly" and set it to 0 if part, 1 if assembly.
#   If the next row has a higher number, than the current line is an assembly.
#   If the next row has the same number, than the current line is a part.
#   If the next row has a lower number, than the current line is a part.
df_tagged = df_raw.copy()
df_tagged["Assembly"] = 0
for i in range(len(df_tagged) - 1):
    if df_tagged.iloc[i, 0] < df_tagged.iloc[i + 1, 0]:
        df_tagged.iloc[i, -1] = 1
    elif df_tagged.iloc[i, 0] == df_tagged.iloc[i + 1, 0]:
        df_tagged.iloc[i, -1] = 0
    else:
        df_tagged.iloc[i, -1] = 0

df_assemblies = df_tagged[df_tagged["Assembly"] == 1].copy()
df_parts = df_tagged[df_tagged["Assembly"] == 0].copy()

# Remove duplicates from the assemblies and parts
df_assemblies = df_assemblies.drop_duplicates(subset=['Title'], keep="first")
df_assemblies.reset_index(drop=True, inplace=True)
df_assemblies.sort_values(by=["Title"], inplace=True)

df_parts = df_parts.drop_duplicates(subset=['Title'], keep="first")
df_parts.reset_index(drop=True, inplace=True)
df_parts.sort_values(by=["Title"], inplace=True)

print(f"{len(df_assemblies)} assemblies found.")
print(f"{len(df_parts)} parts found.")

# df_tagged
# df_assemblies
# df_parts

7 assemblies found.
26 parts found.


In [13]:
# Init PartsBoxAPI

# Add parts to PartsBox and then to assemblies
import sys
from PartsBoxAPI.PartsBoxAPI import PartsBoxAPI
from datetime import datetime
# Add all non-existing parts into partsbox
tags = ["3DExperience"]  # You need to create the tag in PartsBox first for assemblies and parts separately.

logger.remove()
logger.add(sys.stderr, level="ERROR")

# Actual API
link = "https://partsbox.com/pcnzl"
PartsBox = PartsBoxAPI("partsboxapi_6cf6evkbnmhr4a70pqxzd2hcvab3bda5c95114707b905ab70858635cb5cd549d")  # Real API key

# Test API
# link = "https://partsbox.com/chlepcnzl"
# PartsBox = PartsBoxAPI("partsboxapi_8dvrrisdiggkmg1tekt3973dpga6c2exee90d5a4d82ed8964decb742a7ff518c85d9326")  # Test API key


In [14]:
# Add assemblies to assembly database

# Get all assemblies from PartsBox
assemblies = PartsBox.projects.get_all_projects()['data']
assemblies = {assembly["project/name"]: assembly for assembly in assemblies}


# Add all assemblies to PartsBox
for i, row in df_assemblies.iterrows():
    formatted_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if row["Title"] in assemblies.keys():
        current_part = assemblies[row["Title"]]
        if current_part["project/description"] == row["Description"]:
            print(f"Assembly {row['Title']} exists. No changes required. {link}/project/{current_part['project/id']}")
            continue
        part = PartsBox.projects.update_project(
            project_id=assemblies[row["Title"]]["project/id"],
            project_name=row["Title"],
            project_description=row["Description"],
            project_notes=f"{current_part["project/notes"]}\n- {formatted_date}: ({row["Revision"]}) Updated by Python PartsBox API script from 3DExperience CSV output.",
            project_tags=tags,
        )        
        print(f"Assembly {row['Title']} exists. Updated. {link}/project/{part['project/id']}")
    else:
        
        part = PartsBox.projects.create_project(
            project_name=row["Title"],
            project_description=row["Description"],
            project_notes=f"- {formatted_date}: ({row["Revision"]}) Added by Python PartsBox API script from 3DExperience CSV output.",
            project_tags=tags,
        )
        print(f"Adding assembly {row['Title']} to PartsBox. {link}/project/{part['project/id']}")

print(f"All assemblies added. Please go through and check that all parts have sub-assemblies ")

Assembly HLA0001-01 exists. No changes required. https://partsbox.com/pcnzl/project/9vapn0yjg2gpf8zwvmm0waezxk
Assembly PRT0011-01 exists. No changes required. https://partsbox.com/pcnzl/project/1qt1h8njygkdz8bswcwz83ya42
Assembly PRT0013-01 exists. No changes required. https://partsbox.com/pcnzl/project/9pcemxgs52hxsbndsrnd254bt4
Assembly PRT0019-01 exists. No changes required. https://partsbox.com/pcnzl/project/5zk21p0w16hxt8hvqx8b768j2j
Assembly PRT0028-01 exists. No changes required. https://partsbox.com/pcnzl/project/1bas80f3dyjy9ahfx7hrnxvr48
Assembly PRT0032-01 exists. No changes required. https://partsbox.com/pcnzl/project/fn8nkqrzmtkvp8rtwvh4gznf4e
Assembly PRT0035-01 exists. No changes required. https://partsbox.com/pcnzl/project/dc80m5zsfejptb4k9fzr59k6e1
All assemblies added. Please go through and check that all parts have sub-assemblies 


In [15]:
# Add parts to parts database

# Get all parts and assemblies from PartsBox
parts = PartsBox.parts.get_all_parts()['data']
parts = {part["part/name"]: part for part in parts}

# Add all parts to PartsBox
for i, row in df_parts.iterrows():
    formatted_date = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    if row["Title"] in parts.keys():
        current_part = parts[row["Title"]]
        if current_part['part/description'] == row["Description"]:
            print(f"Part {row['Title']} exists. No changes requried. {link}/parts/{parts[row["Title"]]["part/id"]}")
            continue
        part = PartsBox.parts.update_part(
            part_id=parts[row["Title"]]["part/id"],
            part_name=row["Title"],
            part_cad_key=row["Title"],
            part_description=row["Description"],
            part_notes=f"{current_part["part/notes"]}\n- {formatted_date}: ({row["Revision"]}) Updated by Python PartsBox API script from 3DExperience CSV output.",
            part_tags=tags,
        )
        print(f"Part {row['Title']} exists. Updated. {link}/parts/{parts[row['Title']]['part/id']}")
        
    else:
        part = PartsBox.parts.create_part(
            part_type="local",
            part_name=row["Title"],
            part_cad_key=row["Title"],
            part_description=row["Description"],
            part_notes=f"- {formatted_date}: {row["Revision"]} Added by Python PartsBox API script from 3DExperience CSV output.",
            part_tags=tags,
        )
        print(f"Adding part {row['Title']} to PartsBox. {link}/parts/{part['data']['part/id']}")


Part PRT0012-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/12z4c5htb0jm5apqd7vz9t8swm
Part PRT0014-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/2dkb7wp3b2htk974an2svtd6y2
Part PRT0015-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/0qy47pvgdjkvn9339a542r0043
Part PRT0016-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/7awc9nw5bjjhy8m98hrn4bt1sq
Part PRT0017-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/5j4xzg44eeknxb25m8kmtk9ps6
Part PRT0018-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/cqykbn9qpjk62ac0qzqc94vth8
Part PRT0020-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/701qrrh7f6hrv8wrzg34qqqr1w
Part PRT0021-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/btsrj5xfa6jm8aeydy3a896827
Part PRT0022-01 exists. No changes requried. https://partsbox.com/pcnzl/parts/3wbyj86hqwgqs8q1mm5ertqva9
Part PRT0023-01 exists. No changes requried. https://pa